# LogiScan Stage 1 — Gatekeeper Training
## Binary Argument Detector

**What this does:** Trains a model to distinguish logical arguments from non-arguments.
**Uses:** All fallacy examples from merged_fallacies.json as positive + synthetic non-arguments as negative
**Model:** DistilBERT (lightweight, fast CPU inference)
**Output:** `stage1_logical_detector` folder
**Estimated time:** 15-20 minutes on T4x2 GPU (Kaggle)

In [ ]:
# 1. Install
!pip install -q transformers torch scikit-learn tqdm

In [ ]:
# 2. Imports
import json
from collections import Counter
from pathlib import Path

import torch
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import AutoModelForSequenceClassification, AutoTokenizer, get_linear_schedule_with_warmup

print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}, {torch.cuda.device_count()} device(s) else 'CPU'}")

### Upload merged_fallacies.json

In [ ]:
# 3. Load data from Kaggle datasetDATA_PATH = "/kaggle/input/logiscan-merged-fallacies/merged_fallacies.json"print(f"📁 Loading from: {DATA_PATH}")with open(DATA_PATH) as f:    data = json.load(f)# Positive examples: all fallacy texts ARE logical argumentspos_texts = [d['text'] for d in data if len(d['text'].split()) >= 3]# Negative examples: clearly non-argument statementsneg_texts = [    "I really enjoy pizza on Fridays.",    "The weather is nice today.",    "My cat is sleeping on the couch.",    "Let's meet for coffee tomorrow.",    "That movie was really entertaining.",    "I need to do laundry this weekend.",    "The sunset was amazing last night.",    "My friend recommended this restaurant.",    "I'm feeling tired after work.",    "The garden looks beautiful in spring.",    "Traffic was terrible this morning.",    "This song reminds me of summer.",    "I should call my parents this week.",    "The new coffee shop downtown is great.",    "I finished reading that book yesterday.",] * 300  # 4,500 non-argumentsimport randomrandom.shuffle(neg_texts)neg_texts = neg_texts[:len(pos_texts)]texts = pos_texts + neg_textslabels = [1] * len(pos_texts) + [0] * len(neg_texts)print(f"Dataset: {len(texts)} samples")print(f"  Arguments: {sum(labels)}")print(f"  Non-arguments: {len(labels) - sum(labels)}")print(f"  Ratio: {sum(labels)/len(labels):.1%} / {(len(labels)-sum(labels))/len(labels):.1%}")

In [ ]:
# 4. Split
X_train, X_val, y_train, y_val = train_test_split(
    texts, labels, test_size=0.15, random_state=42, stratify=labels
)
print(f"Train: {len(X_train)}, Val: {len(X_val)}")

In [ ]:
# 5. Dataset
class ArgDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx], truncation=True, padding="max_length",
            max_length=self.max_len, return_tensors="pt"
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long),
        }

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
train_ds = ArgDataset(X_train, y_train, tokenizer)
val_ds = ArgDataset(X_val, y_val, tokenizer)
train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64)
print(f"Batches — Train: {len(train_loader)}, Val: {len(val_loader)}")

In [ ]:
# 6. Baseline
majority = Counter(y_val).most_common(1)[0][1] / len(y_val)
print(f"Majority baseline accuracy: {majority:.4f}")
print(f"Your model must beat {majority:.1%}")

In [ ]:
# 7. Model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AutoModelForSequenceClassification.from_pretrained(
    "distilbert-base-uncased", num_labels=2
).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=3e-5)
epochs = 3
total_steps = len(train_loader) * epochs
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=total_steps // 10, num_training_steps=total_steps
)

print(f"Model: {sum(p.numel() for p in model.parameters()):,} params")
print(f"Epochs: {epochs} | Device: {device}")

In [ ]:
# 8. Train
best_acc = 0

for epoch in range(epochs):
    model.train()
    total_loss = 0
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")

    for batch in pbar:
        input_ids = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_ids, attention_mask=mask, labels=labels)
        loss = outputs.loss
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        total_loss += loss.item()
        pbar.set_postfix({"loss": f"{loss.item():.3f}"})

    # Validate
    model.eval()
    preds, truths = [], []
    with torch.no_grad():
        for batch in val_loader:
            out = model(batch["input_ids"].to(device), batch["attention_mask"].to(device))
            p = torch.argmax(out.logits, dim=1).cpu().numpy()
            preds.extend(p)
            truths.extend(batch["label"].numpy())

    acc = accuracy_score(truths, preds)
    f1 = f1_score(truths, preds, average="binary")
    print(f"Epoch {epoch+1}: loss={total_loss/len(train_loader):.4f}, acc={acc:.4f}, f1={f1:.4f}")

    if acc > best_acc:
        best_acc = acc
        Path("stage1_logical_detector").mkdir(exist_ok=True)
        model.save_pretrained("stage1_logical_detector")
        tokenizer.save_pretrained("stage1_logical_detector")
        print(f"  ✅ Saved (acc={acc:.4f})")

print(f"\nBest accuracy: {best_acc:.4f}")

In [ ]:
# 9. Final evaluation
print(classification_report(truths, preds, target_names=["Non-Argument", "Argument"]))

# Test on examples your pipeline will see
tests = [
    ("If it rains, the ground is wet. The ground is wet, therefore it rained.", 1),
    ("I like pizza on Fridays.", 0),
    ("You cannot trust him because he is not a scientist.", 1),
    ("The weather is nice today.", 0),
    ("All humans are mortal. Socrates is human. Therefore Socrates is mortal.", 1),
    ("Let's meet for coffee tomorrow.", 0),
    ("Either you support this or you hate our country.", 1),
    ("That movie was really entertaining.", 0),
]

model.eval()
correct = 0
print("\nTest predictions:")
for text, expected in tests:
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=256).to(device)
    with torch.no_grad():
        prob = torch.softmax(model(**enc).logits, dim=-1)[0][1].item()
    pred = 1 if prob > 0.5 else 0
    match = "✅" if pred == expected else "❌"
    if pred == expected:
        correct += 1
    print(f"  {match} [{prob:.3f}] {'ARG' if pred else 'NON'} — {text[:70]}...")

print(f"\nAccuracy: {correct}/{len(tests)}")

In [ ]:
# 10. Save output to Kaggle working directory!zip -r stage1_logical_detector.zip stage1_logical_detector/print(f"\n✅ Model saved to /kaggle/working/stage1_logical_detector.zip")print("\nTo download: commit the notebook, then download from the output tab.")print("\nOn your machine:")print("  unzip stage1_logical_detector.zip -d models/")